# FuncCraft quick start

This notebook shows two things:

1. How to build one benchmark function from a plain Python `dict`.
2. How to build and use a `BenchmarkSuite` from a plain Python `dict`.

The spec objects are intentionally shown as dictionaries so they can be
edited, printed, and later dumped to YAML/JSON without extra wrappers.


In [1]:
import random
import sys
from pathlib import Path

from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.pyplot as plt
import numpy as np
import textwrap

sys.path.append(str(Path('..').resolve()))

from pyfunccraft import BasicF, BasicFunctionId, BenchmarkFunction, BenchmarkSuite

def zeros(d):
    return [0.0] * d


## One function

The function spec is a nested dictionary. The keys mirror the benchmark
ingredients, but you do not need to construct helper classes yourself.


In [ ]:
func_spec = {
    'dimension': 10,
    'lower_bound': [-100.0] * 10,
    'upper_bound': [100.0] * 10,
    'component_specs': [
        {
            'base_function': 'Sphere',
            'component_dimension': 10,
            'coordinate_transform': {
                'kind': 'identity',
                'dimension': 10,
                'source_point': zeros(10),
                'target_point': zeros(10),
            },
            'value_transform': {
                'kind': 'identity',
            },
        }
    ],
    'composition_spec': {
        'kind': 'single_component',
    },
    'seed': 0,
    'function_class_label': 'C=NONE;P=NONE;T=NONE;G=Sphere',
    'known_global_minimizer': zeros(10),
    'known_global_value': 100.0,
}

f = BenchmarkFunction(func_spec)
print(f)
print('dimension:', f.dimension)
print('label:', f.spec.function_class_label)
print('spec as dict:', f.spec.to_dict())
print('f(x*):', f([f.spec.known_global_minimizer])[0])


## Benchmark suite

Here we generate 100 functions at dimension 2, then plot all of them
in batches of 24 per PDF page with wrapped titles.


In [3]:
suite_spec = {
    'supported_dimensions': '2',
    'base_functions': list(range(24)),
    'base_function_coord_transforms': [
        {'kind': 'rotation', 'probability': 0.3, 'parameters': []},
        {'kind': 'affine', 'probability': 0.7, 'parameters': []},
    ],
    'coord_transforms': [
        {'kind': 'rotation', 'probability': 0.3, 'parameters': []},
        {'kind': 'affine', 'probability': 0.3, 'parameters': []},
        {'kind': 'blockrotation', 'probability': 0.4, 'parameters': []},
    ],
    'value_transforms': [
        {'kind': 'osc', 'probability': 0.8, 'parameters': []},
        {'kind': 'power', 'probability': 0.2, 'parameters': []},
    ],
    'composition_functions': [
        {'kind': 'cpmlwell', 'probability': 0.25, 'parameters': []},
        {'kind': 'cpmsum', 'probability': 0.25, 'parameters': []},
        {'kind': 'dpmsoftmax', 'probability': 0.5, 'parameters': []},
    ],
    'base_functions_for_compositions': [0, 2, 4, 8, 9, 10, 11, 12, 15, 16, 19, 20, 21, 22, 23],
    'requested_number_of_functions': 100,
    'max_number_of_functions': 0,
    'master_seed': 1,
    'lower_bound': -10.0,
    'upper_bound': 10.0,
    'f_opt': 100.0,
    'suite_label': '',
}

suite = BenchmarkSuite(suite_spec, 2)
print('size:', suite.size)
print('dimension:', suite.dimension)
print('max_number_of_functions:', suite.max_number_of_functions)
print('theoretical_max_number_of_functions:', suite.theoretical_max_number_of_functions)

selected_indices = list(range(len(suite)))
grid_x = np.linspace(-10.0, 10.0, 50)   
grid_y = np.linspace(-10.0, 10.0, 50)
xx, yy = np.meshgrid(grid_x, grid_y)
grid_points = [[float(x), float(y)] for x, y in zip(xx.ravel(), yy.ravel())]

def wrap_title(idx, label, width=26):
    return textwrap.fill(f'{idx}: {label}', width=width, break_long_words=False)

num_columns = 4
num_rows = 6
per_page = num_columns * num_rows
pdf_path = Path('benchmark_suite_100_dim2.pdf')

with PdfPages(pdf_path) as pdf:
    for page_start in range(0, len(selected_indices), per_page):
        page_indices = selected_indices[page_start:page_start + per_page]
        fig = plt.figure(figsize=(16, 24))
        for pos, idx in enumerate(page_indices, start=1):
            function = suite.function(idx)
            values = np.asarray(function(grid_points), dtype=float).reshape(xx.shape)
            values = values 
            zmin = float(np.nanpercentile(values, 5))
            zmax = float(np.nanpercentile(values, 95))
            ax = fig.add_subplot(num_rows, num_columns, pos, projection='3d')
            ax.plot_surface(xx, yy, values, cmap='viridis', linewidth=0.3, antialiased=True)
            ax.contour(xx, yy, values, zdir='z', offset=zmin, cmap='viridis', linewidths=0.5)
            ax.set_title(wrap_title(idx, function.spec.function_class_label), fontsize=7)
            ax.set_xlabel('x1', fontsize=7)
            ax.set_ylabel('x2', fontsize=7)
            ax.set_zlabel('f', fontsize=7)
            #ax.set_zlim(zmin, zmax)
            #ax.set_zscale('log')
            ax.tick_params(labelsize=6)
            ax.view_init(elev=28, azim=-135)
        plt.tight_layout()
        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)

print(f'Saved {pdf_path}')
plt.show()


size: 100
dimension: 2
max_number_of_functions: 100
theoretical_max_number_of_functions: 512924325504
Saved benchmark_suite_100_dim2.pdf


## Pure base functions

This page plots the raw base functions directly, without any coordinate
transform or value transform. The layout matches the benchmark-suite
page style, but uses one page for all 28 implemented base functions.


In [ ]:
base_function_names = [
    'Sphere',
    'SumDifferentPowers',
    'Ellipsoidal',
    'BuecheRastrigin',
    'LinearSlope',
    'AttractiveSector',
    'StepEllipsoidal',
    'StepRastrigin',
    'Rosenbrock',
    'Ackley',
    'Rastrigin',
    'Griewank',
    'Schwefel',
    'SharpRidge',
    'DifferentPowers',
    'Weierstrass',
    'SchafferF7',
    'SchafferF7Cond1000',
    'GriewankRosenbrock',
    'Gallagher21',
    'Katsuura',
    'LunacekBiRastrigin',
    'Zakharov',
    'Levy',
]

grid_x = np.linspace(-100.0, 100.0, 50)
grid_y = np.linspace(-100.0, 100.0, 50)
xx, yy = np.meshgrid(grid_x, grid_y)
grid_points = [[float(x), float(y)] for x, y in zip(xx.ravel(), yy.ravel())]

def wrap_title(idx, label, width=26):
    return textwrap.fill(f"{idx}: {label}", width=width, break_long_words=False)

pdf_path = Path('pure_base_functions_dim2.pdf')
num_columns = 4
num_rows = 6
per_page = num_columns * num_rows

with PdfPages(pdf_path) as pdf:
    fig = plt.figure(figsize=(16, 24))
    for pos, base_name in enumerate(base_function_names, start=1):
        function = BasicF(getattr(BasicFunctionId, base_name), 2)
        values = np.asarray(function(grid_points), dtype=float).reshape(xx.shape)
        zmin = float(np.nanpercentile(values, 5))
        zmax = float(np.nanpercentile(values, 95))
        ax = fig.add_subplot(num_rows, num_columns, pos, projection="3d")
        ax.plot_surface(xx, yy, values, cmap="viridis", linewidth=0.3, antialiased=True)
        ax.contour(xx, yy, values, zdir="z", offset=zmin, cmap="viridis", linewidths=0.5)
        ax.set_title(wrap_title(pos - 1, base_name), fontsize=7)
        ax.set_xlabel("x1", fontsize=7)
        ax.set_ylabel("x2", fontsize=7)
        ax.set_zlabel("f", fontsize=7)
        ax.set_zlim(zmin, zmax)
        ax.tick_params(labelsize=6)
        ax.view_init(elev=28, azim=-135)
    plt.tight_layout()
    pdf.savefig(fig, bbox_inches="tight")
    plt.show()
    plt.close(fig)

print(f"Saved {pdf_path}")
